# Esercitazione PySpark: analisi delle vendite 2019-2021

Questo notebook introduce **PySpark da zero** attraverso un caso pratico: analizzare gli ordini di vendita del periodo 2019-2021.

Il percorso è progettato per un workshop di circa **2-3 ore** in Google Colab.

## Obiettivi

Al termine dell'esercitazione saprai:

- creare una `SparkSession`;
- leggere file CSV applicando uno schema esplicito;
- caricare JSON e navigare strutture annidate;
- esplorare, selezionare, filtrare e ordinare i dati;
- gestire valori nulli e creare colonne calcolate;
- unire DataFrame e produrre aggregazioni;
- creare temporary view e interrogarle con Spark SQL;
- usare il catalogo Spark per individuare le view disponibili;
- confrontare DataFrame API e Spark SQL.


## 1. Preparazione dell'ambiente

### Installazione di PySpark

Google Colab include Java, ma PySpark deve essere installato. Fissiamo la versione per rendere l'esercitazione riproducibile. Dopo l'installazione Colab potrebbe richiedere qualche secondo prima dell'importazione.


In [52]:
%pip install -q pyspark==3.5.3


### Importazione delle librerie

Importiamo i tipi necessari per definire lo schema e le funzioni PySpark che useremo. L'alias `F` rende riconoscibili le funzioni Spark nel codice.


In [53]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType,
    DecimalType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)


### Creazione della SparkSession

La `SparkSession` è il punto di ingresso principale per lavorare con DataFrame e Spark SQL. `getOrCreate()` riutilizza una sessione esistente oppure ne crea una nuova.


In [54]:
spark = (
    SparkSession.builder
    .appName("Esercitazione PySpark - Vendite")
    .getOrCreate()
)

print(f"Versione Spark: {spark.version}")


Versione Spark: 3.5.3


### Concetti essenziali: trasformazioni e azioni

Spark applica la **lazy evaluation**:

- le trasformazioni, come `select()` e `filter()`, definiscono un piano di lavoro;
- le azioni, come `show()` e `count()`, avviano realmente il calcolo.

Questa distinzione diventerà visibile negli esempi successivi.


## 2. Caricamento dei file

### Upload dei dataset in Colab

Esegui la cella e seleziona insieme `2019.csv`, `2020.csv`, `2021.csv` e `customer.json`. Se i file sono già presenti nella sessione Colab, non verrà richiesto un nuovo upload.


In [55]:
from google.colab import files

required_files = {"2019.csv", "2020.csv", "2021.csv", "customer.json"}
missing_files = sorted(name for name in required_files if not Path(name).exists())

if missing_files:
    print("Seleziona i file:", ", ".join(missing_files))
    files.upload()
else:
    print("I file richiesti sono già disponibili.")


Seleziona i file: customer.json


Saving customer.json to customer.json


### Verifica dei file

Prima di proseguire controlliamo che tutti i dataset siano disponibili. L'`assert` interrompe subito l'esecuzione e mostra chiaramente gli eventuali file mancanti.


In [56]:
missing_files = sorted(name for name in required_files if not Path(name).exists())
assert not missing_files, f"File mancanti: {missing_files}"

for name in sorted(required_files):
    print(f"{name}: {Path(name).stat().st_size:,} byte")


2019.csv: 123,846 byte
2020.csv: 286,129 byte
2021.csv: 2,898,183 byte
customer.json: 3,052 byte


## 3. Schema e lettura dei CSV

### Definizione dello schema

I file degli ordini **non contengono una riga di intestazione**. Definiamo quindi nomi e tipi delle colonne manualmente.

Per prezzi e imposte utilizziamo `DecimalType` invece di `FloatType`, perché i numeri decimali sono più adatti ai valori monetari.


In [57]:
order_schema = StructType([
    StructField("SalesOrderNumber", StringType(), nullable=False),
    StructField("SalesOrderLineNumber", IntegerType(), nullable=False),
    StructField("OrderDate", DateType(), nullable=False),
    StructField("CustomerName", StringType(), nullable=True),
    StructField("Email", StringType(), nullable=True),
    StructField("Item", StringType(), nullable=False),
    StructField("Quantity", IntegerType(), nullable=False),
    StructField("UnitPrice", DecimalType(12, 4), nullable=False),
    StructField("Tax", DecimalType(12, 4), nullable=False),
])

print(order_schema.simpleString())


struct<SalesOrderNumber:string,SalesOrderLineNumber:int,OrderDate:date,CustomerName:string,Email:string,Item:string,Quantity:int,UnitPrice:decimal(12,4),Tax:decimal(12,4)>


### Funzione riutilizzabile per leggere un anno

La funzione applica sempre le stesse opzioni e aggiunge `OrderYear`, utile per confrontare gli anni. `header=False` è fondamentale: impostandolo a `True` perderemmo il primo ordine di ogni file.


In [58]:
def read_orders(file_name: str, year: int):
    return (
        spark.read
        .option("header", False)
        .option("dateFormat", "yyyy-MM-dd")
        .schema(order_schema)
        .csv(file_name)
        .withColumn("OrderYear", F.lit(year))
    )


### Lettura dei tre dataset

Creiamo un DataFrame distinto per ogni anno. I nomi espliciti evitano di sovrascrivere accidentalmente una variabile generica come `df`.


In [59]:
orders_2019 = read_orders("2019.csv", 2019)
orders_2020 = read_orders("2020.csv", 2020)
orders_2021 = read_orders("2021.csv", 2021)


### Controllo dello schema

`printSchema()` mostra la struttura interpretata da Spark. Verifica in particolare che `OrderDate` sia una data e che prezzi e imposte siano decimali.


In [60]:
orders_2019.printSchema()


root
 |-- SalesOrderNumber: string (nullable = true)
 |-- SalesOrderLineNumber: integer (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: decimal(12,4) (nullable = true)
 |-- Tax: decimal(12,4) (nullable = true)
 |-- OrderYear: integer (nullable = false)



### Visualizzazione delle prime righe

`show()` è un'azione: Spark legge i dati necessari e visualizza un campione. `truncate=False` evita di abbreviare nomi e descrizioni.


In [61]:
orders_2019.show(5, truncate=False)


+----------------+--------------------+----------+--------------+-----------------------------+-----------------------+--------+---------+--------+---------+
|SalesOrderNumber|SalesOrderLineNumber|OrderDate |CustomerName  |Email                        |Item                   |Quantity|UnitPrice|Tax     |OrderYear|
+----------------+--------------------+----------+--------------+-----------------------------+-----------------------+--------+---------+--------+---------+
|SO43701         |1                   |2019-07-01|NULL          |christy12@adventure-works.com|Mountain-100 Silver, 44|1       |3399.9900|271.9992|2019     |
|SO43704         |1                   |2019-07-01|Julio Ruiz    |julio1@adventure-works.com   |Mountain-100 Black, 48 |1       |3374.9900|269.9992|2019     |
|SO43705         |1                   |2019-07-01|Curtis Lu     |curtis9@adventure-works.com  |Mountain-100 Silver, 38|1       |3399.9900|271.9992|2019     |
|SO43700         |1                   |2019-07-01|Ru

### Conteggio degli ordini per anno

`count()` è un'altra azione. I conteggi permettono anche di verificare che la prima riga dei file non sia stata eliminata per errore.


In [62]:
print("2019:", orders_2019.count())
print("2020:", orders_2020.count())
print("2021:", orders_2021.count())


2019: 1201
2020: 2733
2021: 28784


### Esercizio 1: esplorare un DataFrame

Completa la cella per visualizzare dieci righe del 2020 senza troncare il testo e stampare il numero delle colonne.


In [63]:
# ESERCIZIO
# orders_2020. ...
# print(...)


### Soluzione dell'esercizio 1

La proprietà `columns` restituisce una lista Python con i nomi delle colonne; `len()` ne calcola il numero.


In [64]:
orders_2020.show(10, truncate=False)
print("Numero di colonne:", len(orders_2020.columns))


+----------------+--------------------+----------+-----------------+------------------------------+------------------+--------+---------+--------+---------+
|SalesOrderNumber|SalesOrderLineNumber|OrderDate |CustomerName     |Email                         |Item              |Quantity|UnitPrice|Tax     |OrderYear|
+----------------+--------------------+----------+-----------------+------------------------------+------------------+--------+---------+--------+---------+
|SO45347         |1                   |2020-01-01|Clarence Raji    |clarence35@adventure-works.com|Road-650 Black, 52|1       |699.0982 |55.9279 |2020     |
|SO45345         |1                   |2020-01-01|Bonnie Yuan      |bonnie12@adventure-works.com  |Road-150 Red, 52  |1       |3578.2700|286.2616|2020     |
|SO45348         |1                   |2020-01-01|Leah Guo         |leah14@adventure-works.com    |Road-150 Red, 44  |1       |3578.2700|286.2616|2020     |
|SO45349         |1                   |2020-01-01|Candice 

## 4. Unione e prima esplorazione

### Unione dei tre anni

`unionByName()` combina DataFrame con lo stesso schema allineando le colonne per nome. Il risultato rappresenta l'intero periodo 2019-2021.


In [65]:
all_orders = (
    orders_2019
    .unionByName(orders_2020)
    .unionByName(orders_2021)
)

print("Righe complessive:", all_orders.count())


Righe complessive: 32718


### Selezione delle colonne

`select()` crea un nuovo DataFrame senza modificare quello originale. Questa immutabilità consente di costruire trasformazioni progressive e controllabili.


In [66]:
order_summary = all_orders.select(
    "SalesOrderNumber",
    "OrderDate",
    "CustomerName",
    "Item",
    "Quantity",
    "UnitPrice",
    "OrderYear",
)

order_summary.show(5, truncate=False)


+----------------+----------+--------------+-----------------------+--------+---------+---------+
|SalesOrderNumber|OrderDate |CustomerName  |Item                   |Quantity|UnitPrice|OrderYear|
+----------------+----------+--------------+-----------------------+--------+---------+---------+
|SO43701         |2019-07-01|NULL          |Mountain-100 Silver, 44|1       |3399.9900|2019     |
|SO43704         |2019-07-01|Julio Ruiz    |Mountain-100 Black, 48 |1       |3374.9900|2019     |
|SO43705         |2019-07-01|Curtis Lu     |Mountain-100 Silver, 38|1       |3399.9900|2019     |
|SO43700         |2019-07-01|Ruben Prasad  |Road-650 Black, 62     |1       |699.0982 |2019     |
|SO43703         |2019-07-01|Albert Alvarez|Road-150 Red, 62       |1       |3578.2700|2019     |
+----------------+----------+--------------+-----------------------+--------+---------+---------+
only showing top 5 rows



### Filtraggio delle righe

`filter()` mantiene solamente gli ordini che soddisfano una condizione. Qui cerchiamo le righe con prezzo unitario almeno pari a 2.000.


In [67]:
high_price_orders = all_orders.filter(F.col("UnitPrice") >= 2000)
high_price_orders.select("Item", "UnitPrice", "OrderYear").show(10, truncate=False)


+-----------------------+---------+---------+
|Item                   |UnitPrice|OrderYear|
+-----------------------+---------+---------+
|Mountain-100 Silver, 44|3399.9900|2019     |
|Mountain-100 Black, 48 |3374.9900|2019     |
|Mountain-100 Silver, 38|3399.9900|2019     |
|Road-150 Red, 62       |3578.2700|2019     |
|Road-150 Red, 62       |3578.2700|2019     |
|Mountain-100 Silver, 44|3399.9900|2019     |
|Road-150 Red, 44       |3578.2700|2019     |
|Mountain-100 Silver, 44|3399.9900|2019     |
|Road-150 Red, 48       |3578.2700|2019     |
|Road-150 Red, 56       |3578.2700|2019     |
+-----------------------+---------+---------+
only showing top 10 rows



### Ordinamento

`orderBy()` ordina le righe. Con `desc()` richiediamo i prezzi dal maggiore al minore.


In [68]:
all_orders.select("Item", "UnitPrice", "OrderYear").orderBy(
    F.col("UnitPrice").desc()
).show(10, truncate=False)


+----------------+---------+---------+
|Item            |UnitPrice|OrderYear|
+----------------+---------+---------+
|Road-150 Red, 48|3578.2700|2019     |
|Road-150 Red, 52|3578.2700|2020     |
|Road-150 Red, 62|3578.2700|2019     |
|Road-150 Red, 52|3578.2700|2020     |
|Road-150 Red, 52|3578.2700|2019     |
|Road-150 Red, 52|3578.2700|2020     |
|Road-150 Red, 56|3578.2700|2019     |
|Road-150 Red, 52|3578.2700|2020     |
|Road-150 Red, 44|3578.2700|2019     |
|Road-150 Red, 48|3578.2700|2020     |
+----------------+---------+---------+
only showing top 10 rows



### Condizioni multiple

Le condizioni Spark si combinano con `&` (AND), `|` (OR) e `~` (NOT). Ogni condizione deve essere racchiusa tra parentesi.


In [69]:
orders_2021_expensive = all_orders.filter(
    (F.col("OrderYear") == 2021) &
    (F.col("UnitPrice") >= 1000)
)

orders_2021_expensive.show(5, truncate=False)


+----------------+--------------------+----------+--------------+----------------------------+-----------------------+--------+---------+--------+---------+
|SalesOrderNumber|SalesOrderLineNumber|OrderDate |CustomerName  |Email                       |Item                   |Quantity|UnitPrice|Tax     |OrderYear|
+----------------+--------------------+----------+--------------+----------------------------+-----------------------+--------+---------+--------+---------+
|SO49171         |1                   |2021-01-01|Mariah Foster |mariah21@adventure-works.com|Road-250 Black, 48     |1       |2181.5625|174.5250|2021     |
|SO49172         |1                   |2021-01-01|Brian Howard  |brian23@adventure-works.com |Road-250 Red, 44       |1       |2443.3500|195.4680|2021     |
|SO49173         |1                   |2021-01-01|Linda Alvarez |linda19@adventure-works.com |Mountain-200 Silver, 38|1       |2071.4196|165.7136|2021     |
|SO49174         |1                   |2021-01-01|Gina Her

### Esercizio 2: selezione e filtro

Trova gli ordini del 2020 con quantità maggiore di uno. Mostra soltanto numero ordine, prodotto, quantità e prezzo unitario.


In [70]:
# ESERCIZIO
# result = all_orders.filter(...).select(...)
# result.show(truncate=False)


### Soluzione dell'esercizio 2

Prima filtriamo le righe, poi riduciamo le colonne con `select()`. Le trasformazioni possono essere concatenate.


In [71]:
result = (
    all_orders
    .filter((F.col("OrderYear") == 2020) & (F.col("Quantity") > 1))
    .select("SalesOrderNumber", "Item", "Quantity", "UnitPrice")
)

result.show(truncate=False)


+----------------+----+--------+---------+
|SalesOrderNumber|Item|Quantity|UnitPrice|
+----------------+----+--------+---------+
+----------------+----+--------+---------+



## 5. Qualità e pulizia dei dati

### Conteggio dei valori nulli

Per ogni colonna trasformiamo la condizione `isNull()` in 1 oppure 0 e sommiamo i risultati. Questa tecnica produce un rapido profilo di completezza.


In [72]:
null_counts = all_orders.select([
    F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
    for column in all_orders.columns
])

null_counts.show(truncate=False)


+----------------+--------------------+---------+------------+-----+----+--------+---------+---+---------+
|SalesOrderNumber|SalesOrderLineNumber|OrderDate|CustomerName|Email|Item|Quantity|UnitPrice|Tax|OrderYear|
+----------------+--------------------+---------+------------+-----+----+--------+---------+---+---------+
|0               |0                   |0        |2           |0    |0   |0       |0        |0  |0        |
+----------------+--------------------+---------+------------+-----+----+--------+---------+---+---------+



### Ispezione dei clienti mancanti

Prima di correggere i dati è utile osservare le righe interessate. `isNull()` individua i valori realmente nulli.


In [73]:
all_orders.filter(F.col("CustomerName").isNull()).show(10, truncate=False)


+----------------+--------------------+----------+------------+-----------------------------+-----------------------+--------+---------+--------+---------+
|SalesOrderNumber|SalesOrderLineNumber|OrderDate |CustomerName|Email                        |Item                   |Quantity|UnitPrice|Tax     |OrderYear|
+----------------+--------------------+----------+------------+-----------------------------+-----------------------+--------+---------+--------+---------+
|SO43701         |1                   |2019-07-01|NULL        |christy12@adventure-works.com|Mountain-100 Silver, 44|1       |3399.9900|271.9992|2019     |
|SO43710         |1                   |2019-07-02|NULL        |katrina20@adventure-works.com|Road-150 Red, 56       |1       |3578.2700|286.2616|2019     |
+----------------+--------------------+----------+------------+-----------------------------+-----------------------+--------+---------+--------+---------+



### Pulizia dei nomi cliente

Consideriamo mancanti sia i valori nulli sia le stringhe vuote o composte da soli spazi. Li sostituiamo con `Unknown`; per gli altri valori applichiamo `trim()`.


In [74]:
clean_orders = all_orders.withColumn(
    "CustomerName",
    F.when(
        F.col("CustomerName").isNull() |
        (F.trim(F.col("CustomerName")) == ""),
        F.lit("Unknown"),
    ).otherwise(F.trim(F.col("CustomerName")))
)


### Verifica della pulizia

Un controllo dopo la trasformazione rende esplicito il risultato atteso: non devono rimanere clienti nulli o vuoti.


In [75]:
remaining_missing_customers = clean_orders.filter(
    F.col("CustomerName").isNull() |
    (F.trim(F.col("CustomerName")) == "")
).count()

assert remaining_missing_customers == 0
print("Pulizia completata: nessun cliente nullo o vuoto.")


Pulizia completata: nessun cliente nullo o vuoto.


## 6. Colonne calcolate

### Calcolo degli importi

Creiamo tre misure:

- `NetAmount`: quantità moltiplicata per prezzo unitario;
- `TaxAmount`: imposta già presente nel file;
- `TotalAmount`: somma di imponibile e imposta.

Manteniamo valori decimali per evitare approssimazioni tipiche dei numeri floating point.


In [76]:
enriched_orders = (
    clean_orders
    .withColumn("NetAmount", F.col("Quantity") * F.col("UnitPrice"))
    .withColumnRenamed("Tax", "TaxAmount")
    .withColumn("TotalAmount", F.col("NetAmount") + F.col("TaxAmount"))
)

enriched_orders.select(
    "Item", "Quantity", "UnitPrice", "NetAmount", "TaxAmount", "TotalAmount"
).show(5, truncate=False)


+-----------------------+--------+---------+---------+---------+-----------+
|Item                   |Quantity|UnitPrice|NetAmount|TaxAmount|TotalAmount|
+-----------------------+--------+---------+---------+---------+-----------+
|Mountain-100 Silver, 44|1       |3399.9900|3399.9900|271.9992 |3671.9892  |
|Mountain-100 Black, 48 |1       |3374.9900|3374.9900|269.9992 |3644.9892  |
|Mountain-100 Silver, 38|1       |3399.9900|3399.9900|271.9992 |3671.9892  |
|Road-650 Black, 62     |1       |699.0982 |699.0982 |55.9279  |755.0261   |
|Road-150 Red, 62       |1       |3578.2700|3578.2700|286.2616 |3864.5316  |
+-----------------------+--------+---------+---------+---------+-----------+
only showing top 5 rows



### Colonna condizionale

`when().otherwise()` è l'equivalente Spark di una struttura `if/else`. Classifichiamo gli ordini in base al totale della riga.


In [77]:
enriched_orders = enriched_orders.withColumn(
    "OrderCategory",
    F.when(F.col("TotalAmount") >= 2000, "High Value")
    .when(F.col("TotalAmount") >= 500, "Medium Value")
    .otherwise("Standard"),
)

enriched_orders.groupBy("OrderCategory").count().show()


+-------------+-----+
|OrderCategory|count|
+-------------+-----+
| Medium Value| 3735|
|   High Value| 6444|
|     Standard|22539|
+-------------+-----+



### Estrazione di informazioni dalla data

Le funzioni `year()` e `month()` lavorano su colonne di tipo data. Aggiungiamo il mese per le analisi temporali successive.


In [78]:
enriched_orders = (
    enriched_orders
    .withColumn("OrderMonth", F.month("OrderDate"))
    .withColumn("OrderYearFromDate", F.year("OrderDate"))
)

enriched_orders.select("OrderDate", "OrderYear", "OrderYearFromDate", "OrderMonth").show(5)


+----------+---------+-----------------+----------+
| OrderDate|OrderYear|OrderYearFromDate|OrderMonth|
+----------+---------+-----------------+----------+
|2019-07-01|     2019|             2019|         7|
|2019-07-01|     2019|             2019|         7|
|2019-07-01|     2019|             2019|         7|
|2019-07-01|     2019|             2019|         7|
|2019-07-01|     2019|             2019|         7|
+----------+---------+-----------------+----------+
only showing top 5 rows



### Controllo di coerenza dell'anno

Confrontiamo l'anno derivato dal nome del file con quello contenuto nella data. Il controllo deve restituire zero righe incoerenti.


In [79]:
invalid_year_rows = enriched_orders.filter(
    F.col("OrderYear") != F.col("OrderYearFromDate")
).count()

assert invalid_year_rows == 0
print("Anno del file e anno della data sono coerenti.")


Anno del file e anno della data sono coerenti.


### Esercizio 3: creare una colonna

Aggiungi `UnitPriceBand` con i valori `Premium` per prezzi almeno pari a 1.000 e `Regular` negli altri casi.


In [80]:
# ESERCIZIO
# exercise_orders = enriched_orders.withColumn(...)
# exercise_orders.groupBy("UnitPriceBand").count().show()


### Soluzione dell'esercizio 3

La nuova colonna viene aggiunta senza modificare `enriched_orders`; il conteggio permette di controllare la distribuzione delle categorie.


In [81]:
exercise_orders = enriched_orders.withColumn(
    "UnitPriceBand",
    F.when(F.col("UnitPrice") >= 1000, "Premium").otherwise("Regular"),
)

exercise_orders.groupBy("UnitPriceBand").count().show()


+-------------+-----+
|UnitPriceBand|count|
+-------------+-----+
|      Premium| 7866|
|      Regular|24852|
+-------------+-----+



## 7. Aggregazioni

### Fatturato per anno

`groupBy()` definisce i gruppi e `agg()` calcola una o più misure. Arrotondiamo il totale solamente per la presentazione.


In [82]:
sales_by_year = (
    enriched_orders
    .groupBy("OrderYear")
    .agg(
        F.countDistinct("SalesOrderNumber").alias("Orders"),
        F.sum("Quantity").alias("Units"),
        F.round(F.sum("TotalAmount"), 2).alias("Revenue"),
    )
    .orderBy("OrderYear")
)

sales_by_year.show()


+---------+------+-----+-----------+
|OrderYear|Orders|Units|    Revenue|
+---------+------+-----+-----------+
|     2019|  1201| 1201| 4172169.84|
|     2020|  2733| 2733| 6882259.14|
|     2021| 12525|28784|11547835.30|
+---------+------+-----+-----------+



### Prodotti con maggiore fatturato

Raggruppiamo per prodotto, sommiamo quantità e ricavi e ordiniamo il risultato in modo decrescente.


In [83]:
top_products = (
    enriched_orders
    .groupBy("Item")
    .agg(
        F.sum("Quantity").alias("Units"),
        F.round(F.sum("TotalAmount"), 2).alias("Revenue"),
    )
    .orderBy(F.col("Revenue").desc())
)

top_products.show(10, truncate=False)


+-----------------------+-----+----------+
|Item                   |Units|Revenue   |
+-----------------------+-----+----------+
|Road-150 Red, 48       |337  |1302347.15|
|Road-150 Red, 62       |336  |1298482.62|
|Road-150 Red, 52       |302  |1167088.54|
|Road-150 Red, 56       |295  |1140036.82|
|Road-150 Red, 44       |281  |1085933.38|
|Mountain-200 Black, 46 |425  |1000022.23|
|Mountain-200 Black, 42 |388  |912032.31 |
|Mountain-200 Silver, 46|371  |881519.97 |
|Mountain-200 Silver, 38|370  |880356.66 |
|Mountain-200 Black, 38 |366  |863080.17 |
+-----------------------+-----+----------+
only showing top 10 rows



### Clienti con maggiore spesa

Escludiamo `Unknown` dalla classifica per evitare che clienti distinti ma privi di nome vengano considerati come una sola persona.


In [84]:
top_customers = (
    enriched_orders
    .filter(F.col("CustomerName") != "Unknown")
    .groupBy("CustomerName")
    .agg(
        F.countDistinct("SalesOrderNumber").alias("Orders"),
        F.round(F.sum("TotalAmount"), 2).alias("TotalSpent"),
    )
    .orderBy(F.col("TotalSpent").desc())
)

top_customers.show(10, truncate=False)


+------------------+------+----------+
|CustomerName      |Orders|TotalSpent|
+------------------+------+----------+
|Larry Vazquez     |4     |11771.59  |
|Kaitlyn Henderson |4     |11767.92  |
|Nichole Nara      |4     |11746.43  |
|Kate Anand        |4     |11741.82  |
|Margaret He       |4     |11708.52  |
|Lawrence Alonso   |4     |11703.85  |
|Terrance Rodriguez|4     |11695.56  |
|Rosa Hu           |4     |11683.01  |
|Aaron Wright      |4     |11678.72  |
|Clarence Gao      |4     |11663.48  |
+------------------+------+----------+
only showing top 10 rows



### Andamento mensile

Raggruppare per anno e mese evita di sommare insieme, per esempio, gennaio 2019 e gennaio 2020.


In [85]:
monthly_sales = (
    enriched_orders
    .groupBy("OrderYear", "OrderMonth")
    .agg(F.round(F.sum("TotalAmount"), 2).alias("Revenue"))
    .orderBy("OrderYear", "OrderMonth")
)

monthly_sales.show(15)


+---------+----------+----------+
|OrderYear|OrderMonth|   Revenue|
+---------+----------+----------+
|     2019|         7|1008751.05|
|     2019|         8| 561053.67|
|     2019|         9| 554395.83|
|     2019|        10| 606615.99|
|     2019|        11| 796867.01|
|     2019|        12| 644486.28|
|     2020|         1| 663722.57|
|     2020|         2| 626825.48|
|     2020|         3| 756943.04|
|     2020|         4| 687285.33|
|     2020|         5| 746562.29|
|     2020|         6| 524736.82|
|     2020|         7| 589681.59|
|     2020|         8| 392699.33|
|     2020|         9| 434426.48|
+---------+----------+----------+
only showing top 15 rows



### Esercizio 4: valore medio degli ordini

Calcola, per ogni anno, la media di `TotalAmount` e chiamala `AverageLineValue`. Ordina il risultato per anno.


In [86]:
# ESERCIZIO
# average_value_by_year = enriched_orders.groupBy(...).agg(...).orderBy(...)
# average_value_by_year.show()


### Soluzione dell'esercizio 4

Questa media riguarda il valore delle righe d'ordine. Un ordine può contenere più righe: la distinzione è importante nell'interpretazione del risultato.


In [87]:
average_value_by_year = (
    enriched_orders
    .groupBy("OrderYear")
    .agg(F.round(F.avg("TotalAmount"), 2).alias("AverageLineValue"))
    .orderBy("OrderYear")
)

average_value_by_year.show()


+---------+----------------+
|OrderYear|AverageLineValue|
+---------+----------------+
|     2019|         3473.91|
|     2020|         2518.21|
|     2021|          401.19|
+---------+----------------+



## 8. Caricamento e trasformazione di dati JSON

I file JSON possono rappresentare strutture più complesse di una tabella: oggetti annidati, array e campi opzionali.

`customer.json` contiene un array di clienti. Ogni cliente possiede un oggetto `address`, un oggetto `preferences` e un array `orders`.


### Lettura di un JSON multilinea

Poiché l'array JSON è distribuito su più righe, abilitiamo l'opzione `multiline`. In questo primo esempio lasciamo che Spark deduca automaticamente lo schema.


In [88]:
customers = (
    spark.read
    .option("multiline", True)
    .json("customer.json")
)

print("Clienti caricati:", customers.count())


Clienti caricati: 10


### Esplorazione dello schema annidato

`printSchema()` mostra la gerarchia dei campi. `struct` identifica un oggetto annidato, mentre `array` identifica una collezione di valori o oggetti.


In [89]:
customers.printSchema()


root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- zip: string (nullable = true)
 |-- age: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- orders: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- item: string (nullable = true)
 |    |    |-- order_id: string (nullable = true)
 |    |    |-- price: long (nullable = true)
 |-- preferences: struct (nullable = true)
 |    |-- newsletter: boolean (nullable = true)
 |    |-- sms_alerts: boolean (nullable = true)



### Visualizzazione dei documenti

`show()` visualizza gli oggetti annidati in forma compatta. Il parametro `truncate=False` evita di abbreviare gli array degli ordini.


In [90]:
customers.show(5, truncate=False)


+---------------+----+-----------+---------------+-----------------------------------------+--------------+
|address        |age |customer_id|name           |orders                                   |preferences   |
+---------------+----+-----------+---------------+-----------------------------------------+--------------+
|{Rome, 00184}  |25  |1          |Alice Rossi    |[{Laptop, A101, 1200}, {Mouse, A102, 25}]|{true, false} |
|{Milan, 20121} |17  |2          |Bob Verdi      |[{Headphones, B201, 80}]                 |{false, true} |
|{Naples, 80100}|35  |3          |Charlie Bianchi|[]                                       |{true, true}  |
|{Turin, 10121} |NULL|4          |Diana Neri     |[{Book, D301, 15}, {Tablet, D302, 300}]  |{false, false}|
|{Rome, 00184}  |40  |5          |Ethan Russo    |[{Smartphone, E401, 900}]                |{true, false} |
+---------------+----+-----------+---------------+-----------------------------------------+--------------+
only showing top 5 rows



### Accesso ai campi annidati

I campi interni si selezionano con la notazione puntata, per esempio `address.city`. Con `alias()` assegniamo nomi semplici alle colonne risultanti.


In [91]:
customer_profile = customers.select(
    "customer_id",
    "name",
    "age",
    F.col("address.city").alias("city"),
    F.col("address.zip").alias("zip"),
    F.col("preferences.newsletter").alias("newsletter"),
)

customer_profile.show(truncate=False)


+-----------+----------------+----+--------+-----+----------+
|customer_id|name            |age |city    |zip  |newsletter|
+-----------+----------------+----+--------+-----+----------+
|1          |Alice Rossi     |25  |Rome    |00184|true      |
|2          |Bob Verdi       |17  |Milan   |20121|false     |
|3          |Charlie Bianchi |35  |Naples  |80100|true      |
|4          |Diana Neri      |NULL|Turin   |10121|false     |
|5          |Ethan Russo     |40  |Rome    |00184|true      |
|6          |Francesca Conti |29  |Florence|50122|true      |
|7          |Giovanni De Luca|50  |Venice  |30121|false     |
|8          |Hanna Greco     |33  |Bologna |40121|true      |
|9          |Ivan Romano     |22  |Palermo |90133|false     |
|10         |Julia Ferri     |28  |Genoa   |16121|true      |
+-----------+----------------+----+--------+-----+----------+



### Filtraggio mediante un campo annidato

Le espressioni sui campi annidati funzionano come quelle sulle normali colonne. Selezioniamo i clienti che hanno accettato la newsletter.


In [92]:
newsletter_customers = customers.filter(
    F.col("preferences.newsletter") == True
).select("name", F.col("address.city").alias("city"))

newsletter_customers.show(truncate=False)


+---------------+--------+
|name           |city    |
+---------------+--------+
|Alice Rossi    |Rome    |
|Charlie Bianchi|Naples  |
|Ethan Russo    |Rome    |
|Francesca Conti|Florence|
|Hanna Greco    |Bologna |
|Julia Ferri    |Genoa   |
+---------------+--------+



### Espansione di un array con `explode_outer`

Ogni cliente può avere più ordini. `explode_outer()` produce una riga per ciascun elemento dell'array e, a differenza di `explode()`, conserva anche i clienti con array vuoto o nullo.


In [93]:
customer_orders = customers.select(
    "customer_id",
    "name",
    F.col("address.city").alias("city"),
    F.explode_outer("orders").alias("order"),
)

customer_orders.show(truncate=False)


+-----------+----------------+--------+-----------------------+
|customer_id|name            |city    |order                  |
+-----------+----------------+--------+-----------------------+
|1          |Alice Rossi     |Rome    |{Laptop, A101, 1200}   |
|1          |Alice Rossi     |Rome    |{Mouse, A102, 25}      |
|2          |Bob Verdi       |Milan   |{Headphones, B201, 80} |
|3          |Charlie Bianchi |Naples  |NULL                   |
|4          |Diana Neri      |Turin   |{Book, D301, 15}       |
|4          |Diana Neri      |Turin   |{Tablet, D302, 300}    |
|5          |Ethan Russo     |Rome    |{Smartphone, E401, 900}|
|6          |Francesca Conti |Florence|{Shoes, F501, 120}     |
|7          |Giovanni De Luca|Venice  |{Camera, G601, 700}    |
|8          |Hanna Greco     |Bologna |{Watch, H701, 250}     |
|9          |Ivan Romano     |Palermo |{Backpack, I801, 60}   |
|10         |Julia Ferri     |Genoa   |{Desk Lamp, J901, 45}  |
+-----------+----------------+--------+-

### Appiattimento degli oggetti nell'array

Dopo l'espansione, i campi del singolo ordine sono ancora contenuti nella struttura `order`. Li selezioniamo per ottenere un DataFrame tabellare.


In [94]:
flat_customer_orders = customer_orders.select(
    "customer_id",
    "name",
    "city",
    F.col("order.order_id").alias("order_id"),
    F.col("order.item").alias("item"),
    F.col("order.price").alias("price"),
)

flat_customer_orders.show(truncate=False)


+-----------+----------------+--------+--------+----------+-----+
|customer_id|name            |city    |order_id|item      |price|
+-----------+----------------+--------+--------+----------+-----+
|1          |Alice Rossi     |Rome    |A101    |Laptop    |1200 |
|1          |Alice Rossi     |Rome    |A102    |Mouse     |25   |
|2          |Bob Verdi       |Milan   |B201    |Headphones|80   |
|3          |Charlie Bianchi |Naples  |NULL    |NULL      |NULL |
|4          |Diana Neri      |Turin   |D301    |Book      |15   |
|4          |Diana Neri      |Turin   |D302    |Tablet    |300  |
|5          |Ethan Russo     |Rome    |E401    |Smartphone|900  |
|6          |Francesca Conti |Florence|F501    |Shoes     |120  |
|7          |Giovanni De Luca|Venice  |G601    |Camera    |700  |
|8          |Hanna Greco     |Bologna |H701    |Watch     |250  |
|9          |Ivan Romano     |Palermo |I801    |Backpack  |60   |
|10         |Julia Ferri     |Genoa   |J901    |Desk Lamp |45   |
+---------

### Creazione di una view dai dati JSON

Il DataFrame appiattito può essere registrato come temporary view esattamente come un DataFrame proveniente da CSV.


In [95]:
flat_customer_orders.createOrReplaceTempView("customer_orders")
assert spark.catalog.tableExists("customer_orders")
print("Temporary view 'customer_orders' creata.")


Temporary view 'customer_orders' creata.


### Query Spark SQL sulla view JSON

La query usa la view per calcolare numero di acquisti e spesa totale per città. Le righe senza ordini vengono escluse tramite `WHERE order_id IS NOT NULL`.


In [96]:
spark.sql("""
    SELECT
        city,
        COUNT(order_id) AS Orders,
        ROUND(SUM(price), 2) AS TotalSpent
    FROM customer_orders
    WHERE order_id IS NOT NULL
    GROUP BY city
    ORDER BY TotalSpent DESC
""").show(truncate=False)


+--------+------+----------+
|city    |Orders|TotalSpent|
+--------+------+----------+
|Rome    |3     |2125      |
|Venice  |1     |700       |
|Turin   |2     |315       |
|Bologna |1     |250       |
|Florence|1     |120       |
|Milan   |1     |80        |
|Palermo |1     |60        |
|Genoa   |1     |45        |
+--------+------+----------+



### Esercizio JSON

Utilizza la view `customer_orders` per trovare i cinque articoli più costosi. Mostra cliente, città, articolo e prezzo.


In [97]:
# ESERCIZIO
# spark.sql("""
#     SELECT ...
#     FROM customer_orders
#     WHERE ...
#     ORDER BY ...
#     LIMIT 5
# """).show(truncate=False)


### Soluzione dell'esercizio JSON

Escludiamo le righe senza ordine e ordiniamo il prezzo in modo decrescente.


In [98]:
spark.sql("""
    SELECT
        name,
        city,
        item,
        price
    FROM customer_orders
    WHERE order_id IS NOT NULL
    ORDER BY price DESC
    LIMIT 5
""").show(truncate=False)


+----------------+-------+----------+-----+
|name            |city   |item      |price|
+----------------+-------+----------+-----+
|Alice Rossi     |Rome   |Laptop    |1200 |
|Ethan Russo     |Rome   |Smartphone|900  |
|Giovanni De Luca|Venice |Camera    |700  |
|Diana Neri      |Turin  |Tablet    |300  |
|Hanna Greco     |Bologna|Watch     |250  |
+----------------+-------+----------+-----+



## 9. Temporary view e Spark SQL

### Creazione della view

`createOrReplaceTempView()` assegna un nome SQL al DataFrame. La view non duplica i dati: conserva il piano logico necessario a produrli.

Una temporary view è disponibile solamente nella `SparkSession` corrente.


In [99]:
enriched_orders.createOrReplaceTempView("orders")
print("Temporary view 'orders' creata.")


Temporary view 'orders' creata.


### Esplorazione tramite il catalogo

Il catalogo Spark contiene i metadati di tabelle e view accessibili dalla sessione. `listTables()` permette di verificare che `orders` sia stata registrata.


In [100]:
spark.catalog.listTables()


[Table(name='customer_orders', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='orders', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

### Verifica dell'esistenza della view

`tableExists()` è utile prima di eseguire query che dipendono da una tabella o view.


In [101]:
assert spark.catalog.tableExists("orders")
print("La view 'orders' è disponibile nel catalogo della sessione.")


La view 'orders' è disponibile nel catalogo della sessione.


### Prima query Spark SQL

`spark.sql()` esegue una query SQL e restituisce un nuovo DataFrame Spark. Possiamo quindi continuare a utilizzare metodi come `show()` sul risultato.


In [102]:
sql_sample = spark.sql("""
    SELECT
        SalesOrderNumber,
        OrderDate,
        CustomerName,
        Item,
        TotalAmount
    FROM orders
    ORDER BY OrderDate
    LIMIT 10
""")

sql_sample.show(truncate=False)


+----------------+----------+----------------+-----------------------+-----------+
|SalesOrderNumber|OrderDate |CustomerName    |Item                   |TotalAmount|
+----------------+----------+----------------+-----------------------+-----------+
|SO43701         |2019-07-01|Unknown         |Mountain-100 Silver, 44|3671.9892  |
|SO43704         |2019-07-01|Julio Ruiz      |Mountain-100 Black, 48 |3644.9892  |
|SO43705         |2019-07-01|Curtis Lu       |Mountain-100 Silver, 38|3671.9892  |
|SO43700         |2019-07-01|Ruben Prasad    |Road-650 Black, 62     |755.0261   |
|SO43703         |2019-07-01|Albert Alvarez  |Road-150 Red, 62       |3864.5316  |
|SO43697         |2019-07-01|Cole Watson     |Road-150 Red, 62       |3864.5316  |
|SO43699         |2019-07-01|Sydney Wright   |Mountain-100 Silver, 44|3671.9892  |
|SO43702         |2019-07-01|Colin Anand     |Road-150 Red, 44       |3864.5316  |
|SO43698         |2019-07-01|Rachael Martinez|Mountain-100 Silver, 44|3671.9892  |
|SO4

### Aggregazione tramite Spark SQL

La query usa la stessa view per calcolare il fatturato per anno. Il backtick intorno a `Year` non è necessario perché utilizziamo il nome `OrderYear`, non riservato.


In [103]:
sql_sales_by_year = spark.sql("""
    SELECT
        OrderYear,
        COUNT(DISTINCT SalesOrderNumber) AS Orders,
        SUM(Quantity) AS Units,
        ROUND(SUM(TotalAmount), 2) AS Revenue
    FROM orders
    GROUP BY OrderYear
    ORDER BY OrderYear
""")

sql_sales_by_year.show()


+---------+------+-----+-----------+
|OrderYear|Orders|Units|    Revenue|
+---------+------+-----+-----------+
|     2019|  1201| 1201| 4172169.84|
|     2020|  2733| 2733| 6882259.14|
|     2021| 12525|28784|11547835.30|
+---------+------+-----+-----------+



### Confronto tra DataFrame API e SQL

Entrambe le API vengono tradotte in piani Spark. `explain()` permette di osservare il piano fisico scelto dal motore.


In [104]:
print("PIANO DATAFRAME API")
sales_by_year.explain()

print("\nPIANO SPARK SQL")
sql_sales_by_year.explain()


PIANO DATAFRAME API
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [OrderYear#2078 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(OrderYear#2078 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=5005]
      +- HashAggregate(keys=[OrderYear#2078], functions=[sum(Quantity#2066), sum(TotalAmount#2830), count(distinct SalesOrderNumber#2060)])
         +- Exchange hashpartitioning(OrderYear#2078, 200), ENSURE_REQUIREMENTS, [plan_id=5002]
            +- HashAggregate(keys=[OrderYear#2078], functions=[merge_sum(Quantity#2066), merge_sum(TotalAmount#2830), partial_count(distinct SalesOrderNumber#2060)])
               +- HashAggregate(keys=[OrderYear#2078, SalesOrderNumber#2060], functions=[merge_sum(Quantity#2066), merge_sum(TotalAmount#2830)])
                  +- Exchange hashpartitioning(OrderYear#2078, SalesOrderNumber#2060, 200), ENSURE_REQUIREMENTS, [plan_id=4998]
                     +- HashAggregate(keys=[OrderYear#2078, SalesOrderNumber#2060], function

### Query con CTE

Una Common Table Expression (`WITH`) rende più leggibili le query articolate. Calcoliamo prima il fatturato per cliente e poi selezioniamo i primi dieci.


In [105]:
sql_top_customers = spark.sql("""
    WITH customer_sales AS (
        SELECT
            CustomerName,
            COUNT(DISTINCT SalesOrderNumber) AS Orders,
            ROUND(SUM(TotalAmount), 2) AS TotalSpent
        FROM orders
        WHERE CustomerName <> 'Unknown'
        GROUP BY CustomerName
    )
    SELECT CustomerName, Orders, TotalSpent
    FROM customer_sales
    ORDER BY TotalSpent DESC
    LIMIT 10
""")

sql_top_customers.show(truncate=False)


+------------------+------+----------+
|CustomerName      |Orders|TotalSpent|
+------------------+------+----------+
|Larry Vazquez     |4     |11771.59  |
|Kaitlyn Henderson |4     |11767.92  |
|Nichole Nara      |4     |11746.43  |
|Kate Anand        |4     |11741.82  |
|Margaret He       |4     |11708.52  |
|Lawrence Alonso   |4     |11703.85  |
|Terrance Rodriguez|4     |11695.56  |
|Rosa Hu           |4     |11683.01  |
|Aaron Wright      |4     |11678.72  |
|Clarence Gao      |4     |11663.48  |
+------------------+------+----------+



### Creazione di una seconda view

Anche il risultato di una query può diventare una view. Registriamo le vendite mensili per riutilizzarle senza riscrivere l'aggregazione.


In [106]:
monthly_sales.createOrReplaceTempView("monthly_sales")

spark.sql("""
    SELECT *
    FROM monthly_sales
    WHERE OrderYear = 2021
    ORDER BY OrderMonth
""").show()


+---------+----------+----------+
|OrderYear|OrderMonth|   Revenue|
+---------+----------+----------+
|     2021|         1| 572562.09|
|     2021|         2| 480076.26|
|     2021|         3| 570757.62|
|     2021|         4| 583947.97|
|     2021|         5| 624986.67|
|     2021|         6| 931798.85|
|     2021|         7| 915206.59|
|     2021|         8|1116929.33|
|     2021|         9|1141035.00|
|     2021|        10|1360836.40|
|     2021|        11|1802233.99|
|     2021|        12|1447464.55|
+---------+----------+----------+



### Esercizio 5: interrogare una view

Scrivi una query SQL sulla view `orders` che restituisca i cinque prodotti con più unità vendute. Usa `SUM(Quantity)`, `GROUP BY`, `ORDER BY` e `LIMIT`.


In [107]:
# ESERCIZIO
# spark.sql("""
#     SELECT ...
#     FROM orders
#     ...
# """).show(truncate=False)


### Soluzione dell'esercizio 5

L'alias `Units` può essere usato direttamente nella clausola `ORDER BY`.


In [108]:
spark.sql("""
    SELECT
        Item,
        SUM(Quantity) AS Units
    FROM orders
    GROUP BY Item
    ORDER BY Units DESC
    LIMIT 5
""").show(truncate=False)


+---------------------+-----+
|Item                 |Units|
+---------------------+-----+
|Water Bottle - 30 oz.|2097 |
|Patch Kit/8 Patches  |1621 |
|Mountain Tire Tube   |1581 |
|Road Tire Tube       |1212 |
|Sport-100 Helmet, Red|1096 |
+---------------------+-----+



### Rimozione di una view

`dropTempView()` elimina il riferimento dal catalogo, non i dati originali. Manteniamo `orders` per il mini-progetto e rimuoviamo la view intermedia.


In [109]:
removed = spark.catalog.dropTempView("monthly_sales")
print("View rimossa:", removed)
print("monthly_sales esiste ancora?", spark.catalog.tableExists("monthly_sales"))


View rimossa: True
monthly_sales esiste ancora? False


## 10. Mini-progetto conclusivo

Utilizza la view `orders` per creare un report annuale con:

- anno;
- numero di ordini distinti;
- unità vendute;
- fatturato totale;
- valore medio delle righe;
- percentuale di fatturato rispetto all'intero periodo.

Ordina il risultato cronologicamente. Prova a costruire autonomamente la query prima di aprire la soluzione.


### Area di lavoro del mini-progetto

Scrivi la query all'interno della stringa passata a `spark.sql()`.


In [110]:
# MINI-PROGETTO
# final_report = spark.sql("""
#     WITH ...
#     SELECT ...
# """)
# final_report.show()


### Soluzione del mini-progetto

La prima CTE calcola le misure annuali; la seconda calcola il fatturato complessivo. Il `CROSS JOIN` rende disponibile il totale su ogni riga per il calcolo percentuale.


In [111]:
final_report = spark.sql("""
    WITH yearly AS (
        SELECT
            OrderYear,
            COUNT(DISTINCT SalesOrderNumber) AS Orders,
            SUM(Quantity) AS Units,
            SUM(TotalAmount) AS Revenue,
            AVG(TotalAmount) AS AverageLineValue
        FROM orders
        GROUP BY OrderYear
    ),
    overall AS (
        SELECT SUM(Revenue) AS OverallRevenue
        FROM yearly
    )
    SELECT
        y.OrderYear,
        y.Orders,
        y.Units,
        ROUND(y.Revenue, 2) AS Revenue,
        ROUND(y.AverageLineValue, 2) AS AverageLineValue,
        ROUND(y.Revenue / o.OverallRevenue * 100, 2) AS RevenuePercentage
    FROM yearly AS y
    CROSS JOIN overall AS o
    ORDER BY y.OrderYear
""")

final_report.show()


+---------+------+-----+-----------+----------------+-----------------+
|OrderYear|Orders|Units|    Revenue|AverageLineValue|RevenuePercentage|
+---------+------+-----+-----------+----------------+-----------------+
|     2019|  1201| 1201| 4172169.84|         3473.91|            18.46|
|     2020|  2733| 2733| 6882259.14|         2518.21|            30.45|
|     2021| 12525|28784|11547835.30|          401.19|            51.09|
+---------+------+-----+-----------+----------------+-----------------+



### Controlli finali

Verifichiamo automaticamente che il report contenga i tre anni previsti e che le percentuali di fatturato sommino circa a 100.


In [112]:
report_rows = final_report.collect()
report_years = {row["OrderYear"] for row in report_rows}
percentage_total = sum(float(row["RevenuePercentage"]) for row in report_rows)

assert report_years == {2019, 2020, 2021}
assert abs(percentage_total - 100.0) <= 0.1
print("Report finale validato.")


Report finale validato.


## 11. Conclusioni

In questa esercitazione abbiamo costruito un flusso completo:

1. definizione dello schema;
2. lettura e unione dei CSV;
3. esplorazione e pulizia;
4. caricamento e appiattimento di JSON annidati;
5. creazione di misure;
6. aggregazioni con DataFrame API;
7. registrazione e interrogazione di temporary view;
8. consultazione del catalogo Spark;
9. produzione di un report SQL.

### Approfondimenti suggeriti

- window functions;
- join tra dataset;
- lettura e scrittura in formato Parquet o Delta;
- partizionamento e caching;
- cataloghi persistenti in Databricks o Microsoft Fabric.


### Chiusura della sessione

Al termine del lavoro possiamo rimuovere la view e arrestare la sessione. Esegui questa cella soltanto quando non devi più utilizzare i DataFrame del notebook.


In [113]:
for view_name in ("orders", "customer_orders"):
    if spark.catalog.tableExists(view_name):
        spark.catalog.dropTempView(view_name)

spark.stop()
print("SparkSession arrestata.")


SparkSession arrestata.
